# Konversi Model AE+MLP → Format Snort

Notebook ini mengkonversi semua file dari ZIP ke format yang bisa dibaca C++ Snort:

| Input | Output | Dipakai untuk |
|---|---|---|
| `ae_encoder_final.h5` | `ae_encoder_final.tflite` | TFLite C API di Snort |
| `ae_mlp_final.h5` | `ae_mlp_final.tflite` | TFLite C API di Snort |
| `scaler.pkl` | `scaler_params.json` | QuantileTransformer di C++ |
| `feature_names.pkl` | `feature_names.json` | Urutan fitur di C++ |
| `threshold_ae_mlp.pkl` | `thresholds.json` | Threshold di C++ |

---
## Sel 1 — Mount Drive dan Upload ZIP

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

import os, json, zipfile, pickle, shutil
import numpy as np
import joblib
import tensorflow as tf

# Path kerja
WORK_DIR = '/content/ae_mlp_files'
OUT_DIR  = '/content/snort_ready'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUT_DIR,  exist_ok=True)

print('TensorFlow:', tf.__version__)
print('Work dir  :', WORK_DIR)
print('Output dir:', OUT_DIR)
print()
print('Silakan upload file ZIP model_AE_MLP.zip')
uploaded = files.upload()


---
## Sel 2 — Ekstrak ZIP

In [ ]:
# Cari file zip yang diupload
zip_name = [f for f in uploaded.keys() if f.endswith('.zip')][0]
print(f'ZIP ditemukan: {zip_name}')

# Ekstrak
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall(WORK_DIR)
    extracted = zf.namelist()

print('\nFile yang diekstrak:')
for f in sorted(extracted):
    path = os.path.join(WORK_DIR, f)
    size = os.path.getsize(path)/1024**2 if os.path.exists(path) else 0
    print(f'  {f:<40} {size:.2f} MB')


---
## Sel 3 — Verifikasi File

In [ ]:
required = [
    'ae_encoder_final.h5',
    'ae_mlp_final.h5',
    'scaler.pkl',
    'feature_names.pkl',
    'threshold_ae_mlp.pkl',
]

print('Verifikasi file yang dibutuhkan:')
all_ok = True
for fname in required:
    path = os.path.join(WORK_DIR, fname)
    exists = os.path.exists(path)
    size   = os.path.getsize(path)/1024**2 if exists else 0
    status = f'OK  ({size:.2f} MB)' if exists else 'TIDAK ADA'
    print(f'  [{"OK" if exists else "!!!"}] {fname:<35} {status}')
    if not exists:
        all_ok = False

print(f'\n{"Semua file ada — lanjut" if all_ok else "ADA FILE KURANG"}')


---
## Sel 4 — Konversi AE Encoder → TFLite

In [ ]:
print('Loading ae_encoder_final.h5...')
encoder = tf.keras.models.load_model(
    os.path.join(WORK_DIR, 'ae_encoder_final.h5')
)
encoder.summary()

print('\nKonversi ke TFLite...')
converter = tf.lite.TFLiteConverter.from_keras_model(encoder)
converter.optimizations = []  # tidak ada quantization, jaga akurasi
converter.target_spec.supported_types = [tf.float32]
tflite_encoder = converter.convert()

# Simpan
enc_path = os.path.join(OUT_DIR, 'ae_encoder_final.tflite')
with open(enc_path, 'wb') as f:
    f.write(tflite_encoder)

size_h5  = os.path.getsize(os.path.join(WORK_DIR, 'ae_encoder_final.h5'))/1024
size_tfl = os.path.getsize(enc_path)/1024
print(f'ae_encoder_final.h5     : {size_h5:.1f} KB')
print(f'ae_encoder_final.tflite : {size_tfl:.1f} KB')

# Verifikasi: test inference TFLite
print('\nVerifikasi TFLite encoder...')
interp = tf.lite.Interpreter(model_path=enc_path)
interp.allocate_tensors()
in_det  = interp.get_input_details()
out_det = interp.get_output_details()
print(f'  Input shape  : {in_det[0]["shape"]}  (harus [1, 38])')
print(f'  Output shape : {out_det[0]["shape"]} (harus [1, 8])')

# Test dengan dummy input
dummy = np.random.rand(1, 38).astype(np.float32)
interp.set_tensor(in_det[0]['index'], dummy)
interp.invoke()
out = interp.get_tensor(out_det[0]['index'])
print(f'  Test output  : {out}')
print('  Encoder TFLite OK ✓')


---
## Sel 5 — Konversi AE MLP Classifier → TFLite

In [ ]:
print('Loading ae_mlp_final.h5...')
mlp_ae = tf.keras.models.load_model(
    os.path.join(WORK_DIR, 'ae_mlp_final.h5')
)
mlp_ae.summary()

print('\nKonversi ke TFLite...')
converter2 = tf.lite.TFLiteConverter.from_keras_model(mlp_ae)
converter2.optimizations = []
converter2.target_spec.supported_types = [tf.float32]
tflite_mlp = converter2.convert()

# Simpan
mlp_path = os.path.join(OUT_DIR, 'ae_mlp_final.tflite')
with open(mlp_path, 'wb') as f:
    f.write(tflite_mlp)

size_h5  = os.path.getsize(os.path.join(WORK_DIR, 'ae_mlp_final.h5'))/1024
size_tfl = os.path.getsize(mlp_path)/1024
print(f'ae_mlp_final.h5     : {size_h5:.1f} KB')
print(f'ae_mlp_final.tflite : {size_tfl:.1f} KB')

# Verifikasi
print('\nVerifikasi TFLite MLP...')
interp2 = tf.lite.Interpreter(model_path=mlp_path)
interp2.allocate_tensors()
in_det2  = interp2.get_input_details()
out_det2 = interp2.get_output_details()
print(f'  Input shape  : {in_det2[0]["shape"]}  (harus [1, 9])')
print(f'  Output shape : {out_det2[0]["shape"]} (harus [1, 1])')

# Test dengan dummy input 9 dim
dummy9 = np.random.rand(1, 9).astype(np.float32)
interp2.set_tensor(in_det2[0]['index'], dummy9)
interp2.invoke()
out2 = interp2.get_tensor(out_det2[0]['index'])
print(f'  Test output  : {out2}  (harus antara 0-1)')
print('  MLP TFLite OK ✓')


---
## Sel 6 — Verifikasi End-to-End Pipeline TFLite

Test full pipeline: 38 fitur → encoder → 8 latent + recon error → mlp → P(Attack)

In [ ]:
print('Test pipeline lengkap AE+MLP via TFLite...')

# Load AE model untuk hitung reconstruction error
ae_model = tf.keras.models.load_model(
    os.path.join(WORK_DIR, 'ae_model_final.h5')
) if os.path.exists(os.path.join(WORK_DIR, 'ae_model_final.h5')) else None

# Dummy input 38 fitur (sudah ter-scale [0,1])
x_input = np.random.rand(1, 38).astype(np.float32)

# Step 1: Encoder → latent 8 dim
interp.set_tensor(in_det[0]['index'], x_input)
interp.invoke()
z_latent = interp.get_tensor(out_det[0]['index'])  # shape [1, 8]
print(f'Step 1 — Latent space : {z_latent.shape} = {z_latent[0][:4]}...')

# Step 2: Hitung reconstruction error
if ae_model is not None:
    x_recon = ae_model.predict(x_input, verbose=0)
    recon_err = np.mean((x_input - x_recon)**2, axis=1, keepdims=True)
else:
    # Kalau ae_model_final.h5 tidak ada di zip, hitung via encoder+decoder
    # Untuk sekarang pakai dummy
    recon_err = np.array([[0.001]], dtype=np.float32)
    print('  (ae_model_final.h5 tidak ada, recon_err dummy)')
print(f'Step 2 — Recon error  : {recon_err[0][0]:.6f}')

# Step 3: Concat latent + recon error = 9 dim
z_9dim = np.concatenate([z_latent, recon_err], axis=1).astype(np.float32)
print(f'Step 3 — Input MLP    : {z_9dim.shape} = 8 latent + 1 error')

# Step 4: MLP → P(Attack)
interp2.set_tensor(in_det2[0]['index'], z_9dim)
interp2.invoke()
p_attack = interp2.get_tensor(out_det2[0]['index'])[0][0]
print(f'Step 4 — P(Attack)    : {p_attack:.4f}')

threshold = 0.72
label = 'ATTACK' if p_attack > threshold else 'BENIGN'
print(f'Step 5 — Label        : {label} (threshold={threshold})')
print('\nPipeline TFLite end-to-end OK ✓')


---
## Sel 7 — Ekspor Scaler → scaler_params.json

In [ ]:
print('Loading scaler.pkl...')
qt = joblib.load(os.path.join(WORK_DIR, 'scaler.pkl'))
print(f'Type   : {type(qt).__name__}')
print(f'Fitur  : {qt.n_features_in_}')
print(f'Quantiles shape: {qt.quantiles_.shape}')

# Ekspor semua parameter yang dibutuhkan C++
scaler_params = {
    'type'               : 'QuantileTransformer',
    'output_distribution': qt.output_distribution,
    'n_features'         : int(qt.n_features_in_),
    'n_quantiles'        : int(qt.n_quantiles_),
    'quantiles'          : qt.quantiles_.tolist(),
    'references'         : qt.references_.tolist(),
}

scaler_path = os.path.join(OUT_DIR, 'scaler_params.json')
with open(scaler_path, 'w') as f:
    json.dump(scaler_params, f)

size = os.path.getsize(scaler_path)/1024**2
print(f'\nscaler_params.json tersimpan: {size:.2f} MB')
print(f'  n_quantiles : {qt.n_quantiles_}')
print(f'  n_features  : {qt.n_features_in_}')
print(f'  distribution: {qt.output_distribution}')

# Verifikasi: transform dummy data dan bandingkan
print('\nVerifikasi scaler...')
x_dummy = np.random.rand(1, qt.n_features_in_).astype(np.float32)
x_scaled = qt.transform(x_dummy)
print(f'  Input range  : [{x_dummy.min():.4f}, {x_dummy.max():.4f}]')
print(f'  Output range : [{x_scaled.min():.4f}, {x_scaled.max():.4f}] (harus [0,1])')
print('  Scaler params OK ✓')


---
## Sel 8 — Ekspor Feature Names → feature_names.json

In [ ]:
print('Loading feature_names.pkl...')
with open(os.path.join(WORK_DIR, 'feature_names.pkl'), 'rb') as f:
    feature_names = pickle.load(f)

print(f'Jumlah fitur: {len(feature_names)}')
print('Daftar fitur:')
for i, feat in enumerate(feature_names, 1):
    print(f'  {i:2d}. {feat}')

# Simpan sebagai JSON
feat_path = os.path.join(OUT_DIR, 'feature_names.json')
feat_data = {
    'n_features'   : len(feature_names),
    'feature_names': feature_names
}
with open(feat_path, 'w') as f:
    json.dump(feat_data, f, indent=2)

print(f'\nfeature_names.json tersimpan')


---
## Sel 9 — Ekspor Threshold → thresholds.json

In [ ]:
print('Loading threshold_ae_mlp.pkl...')
with open(os.path.join(WORK_DIR, 'threshold_ae_mlp.pkl'), 'rb') as f:
    thr_cfg = pickle.load(f)

print('Isi threshold config:')
for k, v in thr_cfg.items():
    if not isinstance(v, list):
        print(f'  {k:<20} : {v}')

# Buat thresholds.json untuk C++
thresholds = {
    'ae_mlp': {
        'threshold'     : float(thr_cfg['threshold']),
        'n_features'    : int(thr_cfg['n_features']),
        'latent_dim'    : int(thr_cfg.get('latent_dim', 8)),
        'input_dim_mlp' : int(thr_cfg.get('input_dim_mlp', 9)),
        'f1'            : float(thr_cfg.get('f1', 0)),
        'fpr'           : float(thr_cfg.get('fpr', 0)),
        'model_files'   : {
            'encoder'  : 'ae_encoder_final.tflite',
            'ae_model' : 'ae_model_final.tflite',
            'mlp'      : 'ae_mlp_final.tflite',
        }
    }
}

thr_path = os.path.join(OUT_DIR, 'thresholds.json')
with open(thr_path, 'w') as f:
    json.dump(thresholds, f, indent=2)

print(f'\nthresholds.json tersimpan')
print(f'  threshold AE+MLP : {thresholds["ae_mlp"]["threshold"]}')
print(f'  n_features       : {thresholds["ae_mlp"]["n_features"]}')
print(f'  latent_dim       : {thresholds["ae_mlp"]["latent_dim"]}')
print(f'  input_dim_mlp    : {thresholds["ae_mlp"]["input_dim_mlp"]}')


---
## Sel 10 — Konversi AE Model Full → TFLite

AE model full dibutuhkan untuk hitung reconstruction error di C++

In [ ]:
ae_h5_path = os.path.join(WORK_DIR, 'ae_model_final.h5')

if os.path.exists(ae_h5_path):
    print('Loading ae_model_final.h5...')
    ae_full = tf.keras.models.load_model(ae_h5_path)
    ae_full.summary()

    print('\nKonversi ke TFLite...')
    conv_ae = tf.lite.TFLiteConverter.from_keras_model(ae_full)
    conv_ae.optimizations = []
    conv_ae.target_spec.supported_types = [tf.float32]
    tflite_ae = conv_ae.convert()

    ae_tfl_path = os.path.join(OUT_DIR, 'ae_model_final.tflite')
    with open(ae_tfl_path, 'wb') as f:
        f.write(tflite_ae)

    size = os.path.getsize(ae_tfl_path)/1024
    print(f'ae_model_final.tflite tersimpan: {size:.1f} KB')

    # Verifikasi
    interp_ae = tf.lite.Interpreter(model_path=ae_tfl_path)
    interp_ae.allocate_tensors()
    in_ae  = interp_ae.get_input_details()
    out_ae = interp_ae.get_output_details()
    print(f'  Input  : {in_ae[0]["shape"]}  (harus [1, 38])')
    print(f'  Output : {out_ae[0]["shape"]} (harus [1, 38])')
    print('  AE model TFLite OK ✓')
else:
    print('ae_model_final.h5 tidak ada di ZIP')
    print('Reconstruction error akan dihitung dari encoder saja')
    print('(kurang akurat tapi masih bisa jalan)')


---
## Sel 11 — Ringkasan dan ZIP Output untuk Snort

In [ ]:
print('=' * 60)
print(' RINGKASAN FILE OUTPUT UNTUK SNORT')
print('=' * 60)

expected_files = [
    ('ae_encoder_final.tflite', 'Encoder: 38 fitur → 8 latent dim'),
    ('ae_mlp_final.tflite',     'MLP: 9 dim → P(Attack)'),
    ('ae_model_final.tflite',   'AE full: untuk reconstruction error'),
    ('scaler_params.json',      'QuantileTransformer params untuk C++'),
    ('feature_names.json',      '38 nama fitur dalam urutan benar'),
    ('thresholds.json',         'Threshold 0.72 + metadata'),
]

all_ok = True
for fname, desc in expected_files:
    path = os.path.join(OUT_DIR, fname)
    exists = os.path.exists(path)
    size   = os.path.getsize(path)/1024 if exists else 0
    status = f'OK ({size:.1f} KB)' if exists else 'TIDAK ADA'
    print(f'  [{"OK" if exists else "!!!"}] {fname:<35} {status}')
    print(f'       {desc}')
    if not exists:
        all_ok = False

# Buat ZIP
import zipfile
zip_path = '/content/snort_ae_mlp_ready.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname, _ in expected_files:
        path = os.path.join(OUT_DIR, fname)
        if os.path.exists(path):
            zf.write(path, arcname=fname)

zip_size = os.path.getsize(zip_path)/1024**2
print(f'\nsnort_ae_mlp_ready.zip : {zip_size:.2f} MB')
print('\nDownload ZIP di Sel 12')


---
## Sel 12 — Download ZIP

In [ ]:
from google.colab import files
print('Downloading snort_ae_mlp_ready.zip...')
files.download('/content/snort_ae_mlp_ready.zip')
